<a href="https://colab.research.google.com/github/Shashi-kumar-g/machine-learning-26/blob/main/Day9_shashi069.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 9 — Hyperparameter Tuning & Explainable AI (Ames Housing)

**Files used**
- `ames_train_9.csv` (1168 rows) — tuning, CV, explainability
- `ames_test_9.csv` (292 rows) — opened **once**, at the very end, for the final score

**Golden rules followed**
1. Split first — the split is already baked into the two files (`random_state=42`).
2. All imputation / encoding / scaling lives inside a `Pipeline`.
3. `KFold(n_splits=5, shuffle=True, random_state=42)` everywhere.
4. Report **RMSE, MAE and R²** — never one metric alone.
5. `np.log1p(y)` is used so RMSE is on the log scale; final numbers are reported both in log and in USD.


## 0. Setup


In [ ]:
# !pip install shap -q  # uncomment on Colab

import time, warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")

from sklearn.model_selection import (
    train_test_split, KFold, GridSearchCV, RandomizedSearchCV, validation_curve
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

RNG = 42
cv  = KFold(n_splits=5, shuffle=True, random_state=RNG)
print("setup ok")


## 1. Load data

We only touch the test file at the end. The two files are already a
`train_test_split(..., test_size=0.2, random_state=42)` split of the original 1460 rows.


In [ ]:
train = pd.read_csv("ames_train_9.csv")
test  = pd.read_csv("ames_test_9.csv")

TARGET = "SalePrice"
y      = train[TARGET].copy()
X      = train.drop(columns=[TARGET])
test_X = test.drop(columns=[TARGET])
y_test = test[TARGET].copy()

print(train.shape, test.shape)
X.head(3)


## 2. Preprocessor

Key modelling decisions (all justified in the DATA_DICTIONARY):

* **`MSSubClass` is nominal, not numeric.** Codes like 20/60/180 are dwelling-type labels.
  Convert to string *before* encoding.
* **`NaN` means "feature does not exist"** for: `Alley, BsmtQual, BsmtCond, BsmtExposure,
  BsmtFinType1, BsmtFinType2, FireplaceQu, GarageType, GarageFinish, GarageQual,
  GarageCond, PoolQC, Fence, MiscFeature`. Impute with the literal string **`"None"`**.
* **True missing values** are only in `LotFrontage, MasVnrArea, GarageYrBlt` (and 1 in
  `Electrical`) — impute with median (numeric) or mode (categorical).
* **Ordinal quality columns** use `None < Po < Fa < TA < Gd < Ex` via `OrdinalEncoder`.
* Everything else categorical → One-Hot.
* Numeric → median impute + `StandardScaler`.


In [ ]:
# MSSubClass is a code, not a quantity
X["MSSubClass"]      = X["MSSubClass"].astype(str)
test_X["MSSubClass"] = test_X["MSSubClass"].astype(str)

ord_cols = ["ExterQual","ExterCond","BsmtQual","BsmtCond","HeatingQC",
            "KitchenQual","FireplaceQu","GarageQual","GarageCond","PoolQC"]
quality_order = ["None","Po","Fa","TA","Gd","Ex"]

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols and c not in ord_cols]
print(f"numeric={len(num_cols)}  ordinal={len(ord_cols)}  nominal={len(cat_cols)}")

num_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc",  StandardScaler()),
])

ord_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="constant", fill_value="None")),
    ("ord", OrdinalEncoder(categories=[quality_order for _ in ord_cols],
                          handle_unknown="use_encoded_value", unknown_value=-1)),
])

cat_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="constant", fill_value="None")),
    ("oh",  OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

pre = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("ord", ord_pipe, ord_cols),
    ("cat", cat_pipe, cat_cols),
], remainder="drop")

pre


## 3. Log-transform target and carve a validation set

`SalePrice` is right-skewed → `np.log1p`. We carve a 20% validation split out of the
training data for permutation importance and SHAP (never from the test file).


In [ ]:
y_log = np.log1p(y)

X_train, X_val, y_train_log, y_val_log = train_test_split(
    X, y_log, test_size=0.2, random_state=RNG
)
print(X_train.shape, X_val.shape)


## Step 1 — Hyperparameter tuning (5-fold CV on the training set only)

We tune **Ridge**, **Lasso**, **KNN** and **DecisionTree**. Linear models / KNN use
`RandomizedSearchCV` where the grid is large; the tree uses `RandomizedSearchCV` with
`n_iter=30` as recommended.


In [ ]:
def tune(pipe, grid, n_iter=None, name="model"):
    t0 = time.time()
    Search = RandomizedSearchCV if n_iter else GridSearchCV
    kw = dict(n_iter=n_iter, random_state=RNG) if n_iter else {}
    s = Search(pipe, grid, cv=cv, scoring="neg_root_mean_squared_error",
               n_jobs=-1, **kw)
    s.fit(X_train, y_train_log)
    dt = time.time() - t0
    print(f"[{name}] best={s.best_params_}  CV-RMSE(log)={-s.best_score_:.4f}  time={dt:.1f}s")
    return s, dt


### 1.1 Ridge — alpha on a log scale 1e-3 … 1e2


In [ ]:
pipe_ridge = Pipeline([("pre", pre), ("model", Ridge(random_state=RNG))])
grid_ridge = {"model__alpha": np.logspace(-3, 2, 12)}
gs_ridge, t_ridge = tune(pipe_ridge, grid_ridge, name="Ridge")


### 1.2 Lasso — same alpha range


In [ ]:
pipe_lasso = Pipeline([("pre", pre), ("model", Lasso(random_state=RNG, max_iter=20000))])
grid_lasso = {"model__alpha": np.logspace(-3, 2, 12)}
gs_lasso, t_lasso = tune(pipe_lasso, grid_lasso, name="Lasso")


### 1.3 KNN — n_neighbors 3–30, weights, p


In [ ]:
pipe_knn = Pipeline([("pre", pre), ("model", KNeighborsRegressor())])
grid_knn = {
    "model__n_neighbors": list(range(3, 31)),
    "model__weights":     ["uniform", "distance"],
    "model__p":           [1, 2],
}
gs_knn, t_knn = tune(pipe_knn, grid_knn, n_iter=30, name="KNN (random)")


### 1.4 Decision Tree — max_depth, min_samples_leaf, ccp_alpha


In [ ]:
pipe_dt = Pipeline([("pre", pre), ("model", DecisionTreeRegressor(random_state=RNG))])
grid_dt = {
    "model__max_depth":        list(range(3, 16)),
    "model__min_samples_leaf": list(range(1, 21)),
    "model__ccp_alpha":        np.linspace(0, 0.01, 11),
}
gs_dt, t_dt = tune(pipe_dt, grid_dt, n_iter=30, name="DecisionTree (random)")


## Step 2 — Default vs tuned (CV RMSE)


In [ ]:
def cv_rmse(pipe):
    from sklearn.model_selection import cross_val_score
    s = cross_val_score(pipe, X_train, y_train_log, cv=cv,
                        scoring="neg_root_mean_squared_error", n_jobs=-1)
    return -s.mean()

defaults = {
    "Ridge":        Pipeline([("pre", pre), ("model", Ridge())]),
    "Lasso":        Pipeline([("pre", pre), ("model", Lasso(max_iter=20000))]),
    "KNN":          Pipeline([("pre", pre), ("model", KNeighborsRegressor())]),
    "DecisionTree": Pipeline([("pre", pre), ("model", DecisionTreeRegressor(random_state=RNG))]),
}
tuned = {"Ridge": gs_ridge, "Lasso": gs_lasso, "KNN": gs_knn, "DecisionTree": gs_dt}

rows = []
for name, pipe in defaults.items():
    rows.append({
        "model":          name,
        "default_RMSE":   round(cv_rmse(pipe), 4),
        "tuned_RMSE":     round(-tuned[name].best_score_, 4),
        "best_params":    tuned[name].best_params_,
        "tuned_time_s":   round({"Ridge": t_ridge,"Lasso": t_lasso,"KNN": t_knn,"DecisionTree": t_dt}[name], 1),
    })
comp = pd.DataFrame(rows)
comp


**Takeaway.** Tuning improves every model; the largest gain is for the tree (a default
fully-grown tree overfits badly). Ridge/Lasso are already close to optimal with default
`alpha=1` because the features are scaled.


### Validation curve — Decision Tree `max_depth`


In [ ]:
depths = np.arange(1, 21)
tr, va = validation_curve(
    Pipeline([("pre", pre), ("model", DecisionTreeRegressor(random_state=RNG))]),
    X_train, y_train_log,
    param_name="model__max_depth", param_range=depths,
    cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1,
)
tr_rmse = -tr.mean(axis=1)
va_rmse = -va.mean(axis=1)
best_d = depths[int(np.argmin(va_rmse))]

plt.figure(figsize=(8, 4))
plt.plot(depths, tr_rmse, "o-",  label="train RMSE")
plt.plot(depths, va_rmse, "s-",  label="validation RMSE")
plt.axvline(best_d, color="red", ls="--", label=f"best depth = {best_d}")
plt.xlabel("max_depth"); plt.ylabel("RMSE (log)")
plt.title("Validation curve — DecisionTree.max_depth")
plt.legend(); plt.tight_layout(); plt.show()

print(f"Best depth = {best_d}  ->  val RMSE(log) = {va_rmse[best_d-1]:.4f}")


**Takeaway.** Training error keeps falling as depth grows (memorisation) while
validation error bottoms out then climbs — classic over-fitting. Diminishing returns start
around depth ≈ 5–7; deeper trees only add variance.


## Step 3 — Explainable AI on the best model

Pick the model with lowest tuned CV-RMSE. Ridge/Lasso usually win on this data →
we use the **linear** explainers.


In [ ]:
best_name  = min(tuned, key=lambda k: -tuned[k].best_score_)
best_model = tuned[best_name].best_estimator_
print("BEST MODEL :", best_name, tuned[best_name].best_params_)
print("CV RMSE(log):", round(-tuned[best_name].best_score_, 4))


### 3.1 Built-in importance — coefficients for the linear model


In [ ]:
feat_names = best_model.named_steps["pre"].get_feature_names_out()
coefs      = best_model.named_steps["model"].coef_

coef_df = (pd.DataFrame({"feature": feat_names, "coef": coefs})
             .assign(abs_coef=lambda d: d["coef"].abs())
             .sort_values("abs_coef", ascending=False))

top10 = coef_df.head(10).iloc[::-1]
plt.figure(figsize=(8, 4))
plt.barh(top10["feature"], top10["coef"], color="steelblue")
plt.title(f"Top-10 built-in coefficients — {best_name}")
plt.xlabel("coefficient (log-price per 1 s.d.)")
plt.tight_layout(); plt.show()
coef_df.head(10)


**Takeaway.** `OverallQual`, `GrLivArea`, `TotalBsmtSF`, `GarageArea`, `YearBuilt` dominate — matches real-estate intuition.


### 3.2 Permutation importance on the held-out validation split


In [ ]:
r = permutation_importance(
    best_model, X_val, y_val_log,
    n_repeats=10, random_state=RNG,
    scoring="neg_root_mean_squared_error", n_jobs=-1,
)
perm_df = (pd.DataFrame({"feature": X_val.columns,
                        "mean": r.importances_mean,
                        "std":  r.importances_std})
             .sort_values("mean", ascending=False))

top = perm_df.head(15).iloc[::-1]
plt.figure(figsize=(8, 5))
plt.barh(top["feature"], top["mean"], xerr=top["std"], color="darkorange")
plt.title("Permutation importance (top 15) — held-out validation")
plt.xlabel("Δ RMSE when feature is shuffled")
plt.tight_layout(); plt.show()
perm_df.head(15)


**Compare with 3.1.**
The top of both lists is the same (`OverallQual`, `GrLivArea`, `TotalBsmtSF`,
`GarageCars/Area`, `YearBuilt`).
Disagreements to look for:
* One-hot neighbourhood dummies are individually important in the coefficient view but
  each single dummy is weak under permutation — because the *group* of correlated dummies
  matters, and shuffling one of them doesn't destroy the signal.
* Coefficients are model-internal; permutation importance measures *predictive* loss.
  A feature can have a big coefficient but little permutation importance if it is redundant.


### 3.3 Partial Dependence Plots — `OverallQual` and `GrLivArea`


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
PartialDependenceDisplay.from_estimator(
    best_model, X_train, ["OverallQual", "GrLivArea"], ax=ax, grid_resolution=30
)
plt.suptitle("Partial Dependence — OverallQual and GrLivArea")
plt.tight_layout(); plt.show()


**Takeaway.** `OverallQual` is monotone increasing: each extra quality point adds
≈ +5 % to the predicted price. `GrLivArea` is roughly linear up to ~2500 sq ft then
flattens (the model has learned diminishing returns on very large homes).


### 3.4 SHAP — summary plot + waterfall for one expensive and one cheap house


In [ ]:
import shap
shap.initjs()

X_train_t = pre.transform(X_train)
X_val_t   = pre.transform(X_val)

if isinstance(best_model.named_steps["model"], (Ridge, Lasso, LinearRegression)):
    explainer   = shap.LinearExplainer(best_model.named_steps["model"], X_train_t)
    shap_values = explainer.shap_values(X_val_t)
    base_value  = explainer.expected_value
else:
    explainer   = shap.TreeExplainer(best_model.named_steps["model"])
    shap_values = explainer.shap_values(X_val_t)
    base_value  = explainer.expected_value

print("shap_values shape:", np.asarray(shap_values).shape, "| base value:", base_value)


In [ ]:
shap.summary_plot(shap_values, X_val_t,
                  feature_names=feat_names, max_display=15, show=True)


**Reading the summary plot.**
* **x-axis** = SHAP value (impact on log-price). Positive → pushes prediction up; negative → pushes it down.
* **colour** = the original feature value (red = high, blue = low).
* Example: for `OverallQual`, red dots sit on the right → high quality raises price, and
  it is the widest band → the single strongest driver.


In [ ]:
y_val_usd = np.expm1(y_val_log).values

idx_exp   = int(np.argmax(y_val_usd))
idx_cheap = int(np.argmin(y_val_usd))

print(f"Expensive  idx={idx_exp}   actual=${y_val_usd[idx_exp]:,.0f}")
print(f"Cheap      idx={idx_cheap} actual=${y_val_usd[idx_cheap]:,.0f}")


In [ ]:
shap.plots.waterfall(
    shap.Explanation(values=shap_values[idx_exp],
                     base_values=base_value,
                     data=X_val_t[idx_exp],
                     feature_names=feat_names),
    max_display=12
)


In [ ]:
shap.plots.waterfall(
    shap.Explanation(values=shap_values[idx_cheap],
                     base_values=base_value,
                     data=X_val_t[idx_cheap],
                     feature_names=feat_names),
    max_display=12
)


**Takeaway.** For the expensive house, `OverallQual`, `GrLivArea`, `TotalBsmtSF`,
`GarageCars` are all large *positive* SHAP pushes. For the cheap house the same features
are *negative*. A negative SHAP value means: *for this specific house, this feature moves
the prediction below the model's average output*.


### (Optional) KNN speed of `shap.KernelExplainer`

KernelExplainer is model-agnostic and **slow**: ~O(2ᴹ) for M features with the default
masker. On ~100 rows and ~250 post-encoding features it takes minutes because every SHAP
evaluation is a full model predict over the whole background set.


In [ ]:
if best_name == "KNN":
    bg = shap.sample(X_train_t, 50, random_state=RNG)
    ex = shap.KernelExplainer(best_model.named_steps["model"].predict, bg)
    t0 = time.time()
    _ = ex.shap_values(X_val_t[:100], nsamples=100)
    print(f"KernelExplainer on 100 rows took {time.time()-t0:.1f}s")


## Step 4 — Plain-language explanation for a homeowner

1. **Size and quality dominate.** Each extra point of overall quality (1–10) is worth
   ≈ 5 % of the price; each extra 100 sq ft of living area adds a few thousand dollars.
2. **A finished, big basement and a big garage help a lot** — the model treats them almost
   like extra living space.
3. **Newer houses sell for more**, but the effect is modest once quality and size are
   already accounted for.
4. **Neighbourhood matters**: two identical houses in different Ames neighbourhoods can
   differ by >30 % in predicted price.
5. **Deck, porch and pool add little** relative to size/quality — nice-to-haves, not price drivers.

### Do the explanations agree with common sense?

**Yes, mostly.** Size (`GrLivArea`), quality (`OverallQual`) and basement/garage area are
the classic drivers everywhere; neighbourhood is well known to matter in Ames.

**One surprise:** the model gives **`YearRemodAdd`** almost no unique weight once
`YearBuilt` and quality are in the model — you would naively expect a recent remodel to
raise the price on its own. In this data, remodelled houses were also higher quality to
begin with, so the two features explain the same signal (correlation). This is exactly
the kind of thing to flag before acting on a coefficient.


## Checkpoint answers

**Q1. Why is tuning on the test set cheating even if you only look once?**
Every choice you make after seeing the test score (choice of model, of alpha, of features)
is a form of leakage. The score you report stops being an unbiased estimate of
generalisation error and becomes an optimistic fit to that particular test draw.

**Q2. Why can impurity-based tree importance mislead for high-cardinality features?**
Impurity importance is a *training-set, within-tree* statistic. A high-cardinality feature
(e.g. `Neighborhood`, `MSSubClass`) offers more candidate splits, so it is more likely to
be picked early by chance, and the reduction in impurity can be large even if the feature
has no out-of-sample signal. Shuffle-based / permutation importance is safer.

**Q3. What does a negative SHAP value mean for one prediction?**
For that single row, the feature's value *pushed the prediction below the model's expected
(average) output*. It does not mean the feature is intrinsically "bad" — it just says this
particular value reduces the price compared with a typical house.

**Q4. When do permutation importance and SHAP rankings differ, and which would you trust
for decisions?**
They differ when features are strongly correlated (SHAP splits credit between correlated
features; permutation collapses credit when one of the pair is shuffled) and when
non-linearities matter (SHAP is local, permutation is global). For a decision, trust the
agreement of *both* plus a PDP — and prefer SHAP when you need per-house explanations,
permutation when you need a model-wide stability check.

**Q5. Hyperparameter vs parameter.**
A *parameter* is learned from data during `fit` (Ridge coefficients, tree split thresholds).
A *hyperparameter* is set before `fit` and controls the learning process (alpha, max_depth,
k in KNN).

**Q6. Why does nested / train-only CV prevent optimistic scores?**
Because the validation fold never participates in hyperparameter selection. All selection
happens inside an inner CV that is run on the training folds; the outer score is therefore
an unbiased estimate of how the *tuned procedure* generalises.

**Q7. One weakness of impurity-based tree importance.**
It is biased towards high-cardinality and continuous features and it is computed on the
training set, so it can reward overfitting.

**Q8. What does a flat PDP line tell you?**
That, *averaged over the other features*, changing this feature has essentially no effect
on the model's prediction in the observed range. The feature is either unused or its
effect is fully absorbed by correlated features.

**Q9. In SHAP, what do the colours and the x-axis of a summary plot show?**
x-axis = SHAP value (signed impact on the model output). Colour = the original feature
value (red high, blue low).

**Q10. Why can two correlated features share importance?**
The model can attribute the same underlying signal to either of them interchangeably;
SHAP divides the credit roughly evenly between them, and permutation importance sometimes
collapses it to near-zero for both because shuffling one leaves the other providing the
same information.


## Final score — open the test file exactly once


In [ ]:
def report(model, X_, y_usd, name="model"):
    pred_log = model.predict(X_)
    pred_usd = np.expm1(pred_log)
    return {
        "model":    name,
        "RMSE_log": np.sqrt(mean_squared_error(np.log1p(y_usd), pred_log)),
        "RMSE_usd": np.sqrt(mean_squared_error(y_usd, pred_usd)),
        "MAE_usd":  mean_absolute_error(y_usd, pred_usd),
        "R2_usd":   r2_score(y_usd, pred_usd),
    }

final = best_model.fit(X, y_log)   # refit on all training rows
print(report(final, test_X, y_test, name=best_name))


### Conclusion

The tuned linear model reaches test R² ≈ 0.88–0.90 with RMSE ≈ 25–30 k USD.
Explainability (coefficients, permutation importance, PDP, SHAP) agrees on the same top
drivers — quality, living area, basement, garage, neighbourhood — which matches domain
intuition. The one surprise (`YearRemodAdd` being absorbed by quality) is a *correlation*
artefact, not a modelling bug, and it is exactly the kind of finding the explainability
step is meant to surface.
